# SliceLens 1-chain SVI: parametric then pixelated source

This notebook follows the structure of `Herculens/Slicelens/Herculens_3.ipynb`:

1. load the SliceLens F606 data, masks, PSF, grid, and Herculens image model;
2. define and optionally run a 1-chain parametric EPL + shear + MULTI_GAUSSIAN_ELLIPSE lens/source SVI model;
3. use the first-stage mass/lens-light/RMS solution to initialize a pure pixelated-source model;
4. define and optionally run a 1-chain pixelated-source SVI model.

The heavy SVI cells are guarded by `RUN_SVI = False` by default so opening or running the notebook for inspection will not launch inference accidentally.


In [ ]:
# Environment variables that can affect JAX/HDF5-backed workflows must be set before imports.
import os

os.environ.setdefault("HDF5_USE_FILE_LOCKING", "FALSE")
os.environ.setdefault("XLA_PYTHON_CLIENT_PREALLOCATE", "false")
from copy import deepcopy
from pathlib import Path
import sys
import warnings

warnings.simplefilter("ignore")


In [ ]:
PROJECT_ROOT = Path("./").resolve()
GUI_DIR = PROJECT_ROOT
DATA_DIR = (PROJECT_ROOT / ".." / "Data" / "Slicelens").resolve()

if str(GUI_DIR) not in sys.path:
    sys.path.insert(0, str(GUI_DIR))

PROJECT_ROOT, DATA_DIR


In [ ]:
from Tian_infra import import_function

import_function(globals())
jax.config.update("jax_enable_x64", True)
numpyro.enable_x64()

# Keep the notebook's explicit Tian* names while letting Tian_infra provide the imports.
TianLight = Light
TianMass = Mass
TianSVI = SVI


In [ ]:
# Global switches. Keep RUN_SVI=False until you explicitly want to launch inference.
RUN_SVI = False
RUN_POWER_INIT = False

SEED = 100
NUM_CHAINS = 1
MAX_ITER_PARAMETRIC = 10_000
MAX_ITER_PIXELATED = 10_000

DATASET = "F606"
DATASETS = {
    "F606": {
        "image": DATA_DIR / "EHJlens_F606.fits",
        "source_arc_mask": DATA_DIR / "EHJlens_F606_mask.fits",
        "fit_mask_out": DATA_DIR / "EHJlens_F606_mask_out.fits",
        "psf": DATA_DIR / "F606_slice_psf.npy",
        "pix_scale": 0.05,
        "exposure_time": 1200.0,
        "corner_pixel": 10,
        "gain": 1.0,
        "conjugate_points": jnp.asarray([[0.8, -1.1], [0.1, 2.75]], dtype=jnp.float64),
    },
}

N_GAUSS_LENS = 8
N_GAUSS_SOURCE = 4
SIGMA_LIMS_LENS = [0.01, 1.0]
SIGMA_LIMS_SOURCE = [0.01, 1.0]
SOURCE_GRID_SCALE = 0.6
DEFAULT_PIXEL_GRID_SHAPE = 80
SS_FACTOR = 4
HALFRANGE = [0.1, 0.1, 0.1]


In [ ]:
def load_primary_data(config):
    data = fits.getdata(config["image"]).astype(np.float64)
    corner = int(config["corner_pixel"])
    background_pixels = np.vstack([data[:corner, :corner]])
    data = (data - np.mean(background_pixels)) / float(config["gain"])

    source_arc_mask = fits.getdata(config["source_arc_mask"]).astype(bool)
    # This follows Herculens_3.ipynb: the *_mask_out file marks excluded pixels.
    fit_mask = (1 - fits.getdata(config["fit_mask_out"])).astype(bool)

    psf_kernel = np.load(config["psf"]).astype(np.float64)
    if psf_kernel.shape[0] % 2 == 0:
        psf_kernel = psf_kernel[:-1, :]
    if psf_kernel.shape[1] % 2 == 0:
        psf_kernel = psf_kernel[:, :-1]
    psf_kernel = np.clip(psf_kernel, 0.0, None)
    psf_kernel = psf_kernel / np.sum(psf_kernel)

    rms = float(background_pixels.std())
    return data, source_arc_mask, fit_mask, psf_kernel, rms


def build_pixel_grid_noise_psf(data, config, psf_kernel):
    pixel_grid, xgrid, ygrid, x_axis, y_axis, extent, nx, ny = Geometry.get_pixel_grid(
        data,
        float(config["pix_scale"]),
    )
    psf = PSF(psf_type="PIXEL", kernel_point_source=psf_kernel)
    noise = Noise(nx, ny, exposure_time=float(config["exposure_time"]))
    return pixel_grid, xgrid, ygrid, x_axis, y_axis, extent, nx, ny, psf, noise


data_cfg = DATASETS[DATASET]
data, source_arc_mask_np, fit_mask_np, psf_kernel, rms = load_primary_data(data_cfg)
(
    pixel_grid,
    xgrid,
    ygrid,
    x_axis,
    y_axis,
    extent,
    nx,
    ny,
    psf,
    noise,
) = build_pixel_grid_noise_psf(data, data_cfg, psf_kernel)

source_arc_mask = jnp.asarray(source_arc_mask_np, dtype=bool)
fit_mask = jnp.asarray(fit_mask_np, dtype=bool)
conj_points = data_cfg["conjugate_points"]
npix_fit = int(np.asarray(fit_mask_np).sum())

print(f"{DATASET}: data={data.shape}, source_arc_mask={source_arc_mask_np.sum()}, fit_pixels={npix_fit}, rms={rms:.4g}")


In [ ]:
fig, ax = plt.subplots(figsize=(5, 5))
im = ax.imshow(
    data,
    origin="lower",
    cmap="Grays",
    extent=extent,
    norm=colors.SymLogNorm(linthresh=0.01, vmin=max(np.nanpercentile(data, 1), 1e-6), vmax=np.nanpercentile(data, 99.7)),
)
ax.contour(source_arc_mask_np, levels=[0.5], colors="#ff3300", alpha=0.95, linestyles="dashed", origin="lower", extent=extent)
ax.contour(fit_mask_np, levels=[0.5], colors="#9933ff", alpha=0.75, linestyles="dotted", origin="lower", extent=extent)
ax.plot(np.asarray(conj_points)[:, 0], np.asarray(conj_points)[:, 1], "o", alpha=0.7, markersize=3, color="red")
ax.set_title(DATASET)
fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
plt.show()


In [ ]:
# Reuse scheduler and EPL+shear helpers from HerculensGUI/Tian_infra.py.


In [ ]:
# Reuse power-spectrum conversion, value indexing, and initialization helpers from HerculensGUI/Tian_infra.py.


In [ ]:
mass_model = MassModel(["EPL", "SHEAR"])
lens_light_model = LightModel(["MULTI_GAUSSIAN_ELLIPSE"])
source_light_model_parametric = LightModel(["MULTI_GAUSSIAN_ELLIPSE"])

lens_image_parametric = LensImageExtension(
    deepcopy(pixel_grid),
    deepcopy(psf),
    noise_class=noise,
    lens_light_model_class=lens_light_model,
    lens_mass_model_class=mass_model,
    source_model_class=source_light_model_parametric,
    source_arc_mask=source_arc_mask,
    conjugate_points=conj_points,
    kwargs_numerics={"supersampling_factor": SS_FACTOR},
    source_grid_scale=SOURCE_GRID_SCALE,
)


In [ ]:
def model_parametric(data_obs, conj=True, provided_rms=False):
    mass_params = TianMass.EPL_w_shear("Mass model", "1")
    lens_light = TianLight.multi_gauss_light(
        "Lens light",
        "lens",
        N_GAUSS_LENS,
        SIGMA_LIMS_LENS,
        center_low=-0.5,
        center_high=0.5,
    )
    source_light = TianLight.multi_gauss_light("Source light", "source", N_GAUSS_SOURCE, SIGMA_LIMS_SOURCE)

    if conj:
        traced_conj = lens_image_parametric.trace_conjugate_points(mass_params)
        conj_distance = Geometry.reduced_distance_matrix(traced_conj)
        with numpyro.plate(f"Conjugate points - [{conj_distance.shape[0]}]", conj_distance.shape[0]):
            numpyro.sample("conjugate_points", dist.Exponential(1000), obs=conj_distance)

    model_image = lens_image_parametric.model(
        kwargs_lens=mass_params,
        kwargs_source=source_light,
        kwargs_lens_light=lens_light,
        source_add=True,
    )
    numpyro.deterministic("model_image", model_image)

    if provided_rms:
        model_std = jnp.full_like(model_image, rms)
    else:
        background_rms_model = numpyro.sample("RMS", dist.LogUniform(rms * 0.5, rms * 1.5))
        model_var = lens_image_parametric.Noise.C_D_model(model_image, background_rms=background_rms_model)
        model_std = jnp.sqrt(jnp.maximum(model_var, 1e-12))

    with numpyro.plate(f"Data masked - [{npix_fit}]", npix_fit):
        numpyro.sample("obs", dist.Normal(model_image[fit_mask], model_std[fit_mask]), obs=data_obs[fit_mask])


def params2kwargs_parametric(params, fixed_params=None):
    fixed_params = {} if fixed_params is None else fixed_params
    params_full = params | fixed_params
    return {
        "kwargs_lens": TianMass.params2kwargs_EPL_w_shear(params_full, "1"),
        "kwargs_source": TianLight.params2kwargs_multi_gauss_light(params_full, "source", N_GAUSS_SOURCE),
        "kwargs_lens_light": TianLight.params2kwargs_multi_gauss_light(params_full, "lens", N_GAUSS_LENS),
    }


In [ ]:
parametric_state = None
if RUN_SVI:
    parametric_state = TianSVI.run_one_chain_svi(
        model_parametric,
        jnp.asarray(data, dtype=jnp.float64),
        max_iterations=MAX_ITER_PARAMETRIC,
        seed=SEED,
        learning_rate=0.01,
        init_scale=0.05,
    )
    parametric_kwargs = params2kwargs_parametric(parametric_state["median"])
    _ = Plot.plot_loss(parametric_state["losses"], MAX_ITER_PARAMETRIC, color="tab:blue", alpha=0.8)
else:
    print("RUN_SVI=False: parametric SVI is defined but not executed.")


In [ ]:
parametric_kwargs = params2kwargs_parametric(parametric_state["median"])
pixel_grid_shape = Geometry.get_best_pixel_size(lens_image_parametric, parametric_kwargs, SOURCE_GRID_SCALE)
k_grid = PowerSpectrum.K_grid((pixel_grid_shape, pixel_grid_shape))

source_light_model_pixelated = LightModel(
    ["PIXELATED"],
    pixel_adaptive_grid=True,
    pixel_interpol="fast_bilinear",
    kwargs_pixelated={"num_pixels": pixel_grid_shape},
)

lens_image_pixelated = LensImageExtension(
    deepcopy(pixel_grid),
    deepcopy(psf),
    noise_class=noise,
    lens_light_model_class=LightModel(["MULTI_GAUSSIAN_ELLIPSE"]),
    lens_mass_model_class=MassModel(["EPL", "SHEAR"]),
    source_model_class=source_light_model_pixelated,
    source_arc_mask=jnp.ones_like(jnp.asarray(data), dtype=bool),
    conjugate_points=conj_points,
    kwargs_numerics={"supersampling_factor": SS_FACTOR},
    source_grid_scale=SOURCE_GRID_SCALE,
)

pixel_grid_shape


In [ ]:
PIXELATED_PRIOR = {
    "n_value": None,
    "sigma_low": 1e-5,
    "sigma_high": 10.0,
    "positive": True,
    "k_zero": None,
}


def model_pixelated_source(data_obs, k_values, conj=True):
    mass_params = TianMass.EPL_w_shear("Mass model", "1")
    lens_light = TianLight.multi_gauss_light(
        "Lens light",
        "lens",
        N_GAUSS_LENS,
        SIGMA_LIMS_LENS,
        center_low=-0.5,
        center_high=0.5,
    )

    source_light = [
        PowerSpectrum.matern_power_spectrum(
            "Source grid",
            "source_grid",
            k_values,
            k_zero=PIXELATED_PRIOR["k_zero"],
            n_value=PIXELATED_PRIOR["n_value"],
            sigma_low=PIXELATED_PRIOR["sigma_low"],
            sigma_high=PIXELATED_PRIOR["sigma_high"],
            positive=PIXELATED_PRIOR["positive"],
        )
    ]

    if conj:
        traced_conj = lens_image_pixelated.trace_conjugate_points(mass_params)
        conj_distance = Geometry.reduced_distance_matrix(traced_conj)
        with numpyro.plate(f"Conjugate points - [{conj_distance.shape[0]}]", conj_distance.shape[0]):
            numpyro.sample("conjugate_points", dist.Exponential(1000), obs=conj_distance)

    model_image = lens_image_pixelated.model(
        kwargs_lens=mass_params,
        kwargs_source=source_light,
        kwargs_lens_light=lens_light,
        source_add=True,
    )
    numpyro.deterministic("model_image", model_image)

    background_rms_model = numpyro.sample("RMS", dist.LogUniform(rms * 0.5, rms * 1.5))
    model_var = lens_image_pixelated.Noise.C_D_model(model_image, background_rms=background_rms_model)
    model_std = jnp.sqrt(jnp.maximum(model_var, 1e-12))

    with numpyro.plate(f"Data masked - [{npix_fit}]", npix_fit):
        numpyro.sample("obs", dist.Normal(model_image[fit_mask], model_std[fit_mask]), obs=data_obs[fit_mask])


def params2kwargs_pixelated(params, k_values, fixed_params=None):
    fixed_params = {} if fixed_params is None else fixed_params
    params_full = params | fixed_params
    return {
        "kwargs_lens": TianMass.params2kwargs_EPL_w_shear(params_full, "1"),
        "kwargs_source": [
            PowerSpectrum.params2kwargs_power_spectrum(
                params_full,
                "source_grid",
                k_values,
                positive=PIXELATED_PRIOR["positive"],
                n_value=PIXELATED_PRIOR["n_value"],
                k_zero=PIXELATED_PRIOR["k_zero"],
            )
        ],
        "kwargs_lens_light": TianLight.params2kwargs_multi_gauss_light(params_full, "lens", N_GAUSS_LENS),
    }


In [ ]:
def fit_power_spectrum_init_from_parametric_source(parametric_kwargs, seed=SEED + 7919):
    if parametric_kwargs is None:
        return {}

    source_image, _ = Plot.pixelize_plane(
        lens_image_parametric,
        parametric_kwargs,
        pixel_grid_shape,
        source_grid_scale=SOURCE_GRID_SCALE,
    )
    return PowerSpectrum.fit_power_spectrum_init(
        source_image,
        k_grid.k,
        PIXELATED_PRIOR,
        seed=seed,
        max_iterations=1200,
        learning_rate=0.02,
        noise_factor=0.001,
        progress_bar=True,
    )


pixelated_init_values = {}
if parametric_state is not None:
    pixelated_init_values.update(ResumeInit.pixelated_stage_init_from_parametric(parametric_state["median"]))

if RUN_SVI and RUN_POWER_INIT and parametric_kwargs is not None:
    pixelated_init_values.update(fit_power_spectrum_init_from_parametric_source(parametric_kwargs))

pixelated_state = None
if RUN_SVI:
    pixelated_state = TianSVI.run_one_chain_svi(
        model_pixelated_source,
        jnp.asarray(data, dtype=jnp.float64),
        max_iterations=MAX_ITER_PIXELATED,
        seed=SEED + 1,
        init_values=pixelated_init_values or None,
        learning_rate=0.001,
        init_scale=0.01,
        model_args=(k_grid.k,),
    )
    pixelated_kwargs = params2kwargs_pixelated(pixelated_state["median"], k_grid.k)
    _ = Plot.plot_loss(pixelated_state["losses"], MAX_ITER_PIXELATED, color="tab:orange", alpha=0.8)
else:
    print("RUN_SVI=False: pure pixelated-source SVI is defined but not executed.")


In [ ]:
def _positive_log_panel(image, mask=None, q=(0.5, 99.7)):
    image = np.asarray(image, dtype=float)
    valid = np.isfinite(image) & (image > 0.0)
    if mask is not None:
        valid &= np.asarray(mask, dtype=bool)
    panel = np.ma.array(image, mask=~valid)
    if panel.count() == 0:
        return panel, None
    vmin, vmax = np.nanpercentile(panel.compressed(), q)
    vmin = max(float(vmin), np.finfo(float).tiny)
    vmax = max(float(vmax), vmin * 1.01)
    return panel, colors.LogNorm(vmin=vmin, vmax=vmax)


def _source_image_from_kwargs(lens_image, kwargs, label):
    kwargs_source = kwargs.get("kwargs_source", [])
    if kwargs_source and "pixels" in kwargs_source[0]:
        return np.asarray(kwargs_source[0]["pixels"], dtype=float)

    # Parametric-source runs are shown on the same pixel grid used to initialize the pixelated source.
    source_image, _ = Plot.pixelize_plane(
        lens_image,
        kwargs,
        pixel_grid_shape,
        source_grid_scale=SOURCE_GRID_SCALE,
    )
    return np.asarray(source_image, dtype=float)


def visualize_model_four_panel(lens_image, kwargs, data_obs, title="model visualization"):
    model_image = lens_image.model(**kwargs)
    model_var = lens_image.Noise.C_D_model(model_image, background_rms=rms)
    residual = (jnp.asarray(data_obs) - model_image) / jnp.sqrt(jnp.maximum(model_var, 1e-12))
    source_image = _source_image_from_kwargs(lens_image, kwargs, title)

    fig, axes = plt.subplots(1, 4, figsize=(18, 4.5), constrained_layout=True)
    panels = [
        (data_obs, "data", "twilight", "log"),
        (np.asarray(model_image), "model", "twilight", "log"),
        (np.asarray(residual), "(data - model) / rms", "bwr", "residual"),
        (source_image, "source image", "twilight", "source"),
    ]

    for ax, (image, label, cmap, mode) in zip(axes, panels):
        if mode == "residual":
            panel = np.ma.array(image, mask=(~fit_mask) | (~np.isfinite(image)))
            im = ax.imshow(panel, origin="lower", extent=extent, cmap=cmap, vmin=-3, vmax=3)
            fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
        elif mode == "source":
            panel, norm = _positive_log_panel(image)
            im = ax.imshow(panel, origin="lower", cmap=cmap, norm=norm)
            if norm is None:
                fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
        else:
            panel, norm = _positive_log_panel(image, fit_mask)
            im = ax.imshow(panel, origin="lower", extent=extent, cmap=cmap, norm=norm)
        ax.set_title(label)
        if mode != "source":
            ax.set_xlabel("arcsec")
            ax.set_ylabel("arcsec")

    fig.suptitle(title, y=1.04)
    return fig


if RUN_SVI and parametric_state is not None:
    visualize_model_four_panel(lens_image_parametric, parametric_kwargs, data, title="parametric source")

if RUN_SVI and pixelated_state is not None:
    visualize_model_four_panel(lens_image_pixelated, pixelated_kwargs, data, title="pixelated source")
else:
    print("No pixelated preview generated because SVI was not run in this session.")